# Subclustering with Harmony

Selected method excerpts from a supplied exploratory notebook. Project attribution is not established, so this notebook makes no claim about which project used these steps or about biological results. Original annotations, marker panels, private paths and outputs are withheld.

The retained method sequence uses an existing PCA representation, applies Harmony by a batch field, recomputes neighbors, and runs UMAP and Leiden subclustering. The public copy adds an explicit before/after graph comparison and input checks. These adaptations are labeled below.

Run from the repository root with Scanpy, harmonypy, NumPy, pandas, Matplotlib, igraph and leidenalg installed. Input: `data/private/subclustering/annotated.h5ad`, including precomputed `X_pca`, a boolean `subset_selected` field, and a `batch` field. The input must already have an appropriate normalized/scaled PCA representation. Code syntax and notebook format were checked; no private-data execution or scientific validation is claimed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib import rcParams

# Run from the repository root. These local inputs and outputs are not distributed.
PRIVATE_DIR = Path("data/private/subclustering")
OUTPUT_DIR = Path("outputs/subclustering/python")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = OUTPUT_DIR
sc.settings.verbosity = 3
sc.settings.set_figure_params(scanpy=True, dpi=80, dpi_save=300,
                             frameon=True, vector_friendly=True, fontsize=14)
rcParams['figure.figsize'] = 8, 8

In [ ]:
adata = sc.read_h5ad(PRIVATE_DIR / 'annotated.h5ad')
# The private input supplies a boolean subset selector and a neutral batch field.
adata_subset = adata[adata.obs['subset_selected'].astype(bool)].copy()

In [ ]:
# Public adaptation: validate the existing PCA representation and compare
# the uncorrected neighbor graph with the corrected graph using the same settings.
if 'X_pca' not in adata_subset.obsm:
    raise ValueError('Supply a precomputed PCA representation.')
if adata_subset.obsm['X_pca'].shape[1] < 39:
    raise ValueError('This excerpt expects at least 39 principal components.')
if adata_subset.obs['batch'].isna().any():
    raise ValueError('Batch labels must be complete.')
if adata_subset.obs['batch'].nunique() < 2:
    raise ValueError('Harmony batch integration requires multiple batches here.')
adata_subset.obsm['X_pca_uncorrected'] = adata_subset.obsm['X_pca'].copy()

rcParams['figure.figsize'] = 8, 8
sc.pp.neighbors(adata_subset, knn=True, n_pcs=39)
sc.tl.umap(adata_subset)
sc.tl.leiden(adata_subset, resolution=0.5)
sc.pl.umap(adata_subset, wspace=0.2, size=30, color=['leiden', 'batch'],
           save='_before-harmony.png')

In [ ]:
# Source-derived Harmony step: correct PCA coordinates by batch.
sc.external.pp.harmony_integrate(adata_subset, 'batch')
adata_subset.obsm['X_pca'] = adata_subset.obsm['X_pca_harmony']

rcParams['figure.figsize'] = 8, 8
sc.pp.neighbors(adata_subset, knn=True, n_pcs=39)
sc.tl.umap(adata_subset)
sc.tl.leiden(adata_subset, resolution=0.5)
sc.pl.umap(adata_subset, wspace=0.2, size=30, color=['leiden', 'batch'],
           save='_after-harmony.png')

In [ ]:
sc.tl.rank_genes_groups(adata_subset, 'leiden', method='logreg')
sc.pl.rank_genes_groups(adata_subset, n_genes=50, sharey=False)